#### 1. 1차 필터
- 룰 검출 문장 중 과거 오류 이력 개체명(`EXT_E001`)이 포함된 문장만 추출
- 계엄 문장(`EXT_S001`)은 휴먼 검수 후보에서 제외
- 대상 문장이 하나라도 있는 기사를 `doc_id` 기준으로 원천 데이터에서 추출

In [4]:
import json
from pathlib import Path

import pandas as pd

from src.data_processing.rule_pipeline import load_refer_data
from src.data_processing.prepare.transform import entity_data, sentence_data
from src.data_processing.rules.entity_rules import (
    ext_e001,
    ext_e002,
    ext_e003,
    ext_e004,
    ext_e005,
    ext_e006,
)
from src.data_processing.rules.sentence_rules import ext_s001, ext_s002, ext_s003

DATA_DIR = Path(r"C:\work\ner\data\org\all_pass_ner_tagged_1")
REPORT_DIR = DATA_DIR / "reports"
input_files = sorted(DATA_DIR.glob("*.json"))
input_files

[WindowsPath('C:/work/ner/data/org/all_pass_ner_tagged_1/강원일보.json'),
 WindowsPath('C:/work/ner/data/org/all_pass_ner_tagged_1/경기일보.json'),
 WindowsPath('C:/work/ner/data/org/all_pass_ner_tagged_1/경향신문.json')]

In [5]:
input_files = input_files[:1]

In [6]:
def load_reviewed_ids(review_path):
    reviewed_doc_ids = set()
    reviewed_sentence_ids = set()
    with review_path.open(encoding="utf-8") as file:
        for line in file:
            row = json.loads(line)
            if row.get("id"):
                reviewed_doc_ids.add(row["id"])
            sentence_ids = row.get("reverted_sentence_ids", [])
            if not sentence_ids:
                sentence_ids = [
                    sentence.get("sentence_id")
                    for sentence in row.get("original_sentences", [])
                ]
            reviewed_sentence_ids.update(
                sentence_id for sentence_id in sentence_ids if sentence_id
            )
    return reviewed_doc_ids, reviewed_sentence_ids


def get_rule_columns(output_data):
    return [
        column
        for column in output_data.columns
        if column.startswith(("EXT_S", "EXT_E"))
        and not column.endswith("_PATTERN")
    ]


def select_target_documents(source_path, target_doc_ids):
    source = json.loads(source_path.read_text(encoding="utf-8-sig"))
    return {
        **{key: value for key, value in source.items() if key != "document"},
        "document": [
            document for document in source["document"]
            if document["id"] in target_doc_ids
        ],
    }


- 룰 검출 결과 생성
- `run_pipeline()`은 사용하지 않고, 필요한 변환·문장 룰·개체명 룰 함수만 직접 실행합니다.
- 따라서 LLM 입력용 재구성 및 중간 pickle 저장은 수행하지 않습니다.

In [10]:
refer_data = load_refer_data()
error_data = refer_data["error_data"]

error_data = refer_data["error_data"]

for quartile in ("Q4", "Q3", "Q2", "Q1"):
    high_frequency_error_data = error_data.loc[
        error_data["freq_quartile"].eq(quartile)
    ].copy()

    if not high_frequency_error_data.empty:
        selected_quartile = quartile
        break
else:
    raise ValueError(
        "고빈도 과거 오류 개체명이 없습니다. "
        "error_data의 freq_quartile을 확인하세요."
    )

print(f"과거 오류 개체명 기준: {selected_quartile}")

과거 오류 개체명 기준: Q3


In [12]:
def run_required_rules(input_path):
    output_data = sentence_data(input_path)
    for rule in (ext_s001, ext_s002, ext_s003):
        output_data = rule(output_data)

    output_data = entity_data(output_data)
    output_data = ext_e001(output_data, high_frequency_error_data)
    output_data = ext_e002(output_data)
    output_data = ext_e003(output_data, refer_data["mistag_data"])
    output_data = ext_e004(output_data, refer_data["tag_data"])
    output_data = ext_e005(output_data, refer_data["tag_data"])
    output_data = ext_e006(output_data, refer_data["tag_data"])
    return output_data

rule_results = {
    input_path.stem: run_required_rules(input_path)
    for input_path in input_files
}

In [13]:
rule_results

{'강원일보':                     id                doc_id               sentence_id  \
 0       NLRW2600000001      NLRW2600000001.1      NLRW2600000001.1.1.1   
 1       NLRW2600000001      NLRW2600000001.1      NLRW2600000001.1.2.1   
 2       NLRW2600000001      NLRW2600000001.1      NLRW2600000001.1.2.1   
 3       NLRW2600000001      NLRW2600000001.1      NLRW2600000001.1.3.1   
 4       NLRW2600000001      NLRW2600000001.1      NLRW2600000001.1.4.1   
 ...                ...                   ...                       ...   
 359666  NLRW2600000001  NLRW2600000001.25051  NLRW2600000001.25051.5.2   
 359667  NLRW2600000001  NLRW2600000001.25051  NLRW2600000001.25051.5.2   
 359668  NLRW2600000001  NLRW2600000001.25051  NLRW2600000001.25051.5.3   
 359669  NLRW2600000001  NLRW2600000001.25051  NLRW2600000001.25051.5.3   
 359670  NLRW2600000001  NLRW2600000001.25051  NLRW2600000001.25051.5.4   
 
                                                      form  EXT_S001  EXT_S002  \
 0      

In [14]:
{press: result.shape for press, result in rule_results.items()}

{'강원일보': (359671, 106)}

In [15]:
def get_target_doc_ids(output_data, review_path):
    rule_columns = get_rule_columns(output_data)
    rule_hit_mask = output_data[rule_columns].eq(True).any(axis=1)
    martial_law_mask = output_data["EXT_S001"].eq(True)
    error_history_mask = output_data["EXT_E001"].eq(True)

    # 룰 검출 건을 문장 단위로 좁힌 뒤, 계엄 문장은 휴먼 검수 후보에서 제외합니다.
    target_sentence_mask = (
        rule_hit_mask & error_history_mask & ~martial_law_mask
    )
    target_sentence_ids = set(
        output_data.loc[target_sentence_mask, "sentence_id"]
    )
    source_doc_count = output_data["doc_id"].nunique()
    source_sentence_count = output_data["sentence_id"].nunique()
    candidate_doc_ids = set(
        output_data.loc[target_sentence_mask, "doc_id"]
    )
    rule_doc_ids = set(output_data.loc[rule_hit_mask, "doc_id"])
    martial_law_doc_ids = set(
        output_data.loc[martial_law_mask, "doc_id"]
    )
    martial_law_sentence_ids = set(
        output_data.loc[martial_law_mask, "sentence_id"]
    )
    error_history_doc_ids = set(
        output_data.loc[error_history_mask, "doc_id"]
    )

    reviewed_doc_ids, reviewed_sentence_ids = load_reviewed_ids(review_path)
    reviewed_candidate_doc_ids = candidate_doc_ids & reviewed_doc_ids

    return {
        "rule_doc_ids": rule_doc_ids,
        "error_history_doc_ids": error_history_doc_ids,
        "candidate_doc_ids": candidate_doc_ids,
        "target_sentence_ids": target_sentence_ids,
        "target_sentence_count": len(target_sentence_ids),
        "source_doc_count": source_doc_count,
        "source_sentence_count": source_sentence_count,
        "martial_law_doc_ids": martial_law_doc_ids,
        "martial_law_sentence_ids": martial_law_sentence_ids,
        "reviewed_doc_ids": reviewed_doc_ids,
        "reviewed_sentence_ids": reviewed_sentence_ids,
        "reviewed_candidate_doc_ids": reviewed_candidate_doc_ids,
    }

doc_ids_by_press = {}
for press, output_data in rule_results.items():
    review_path = REPORT_DIR / press / "review_needed.jsonl"
    doc_ids_by_press[press] = get_target_doc_ids(output_data, review_path)

for ids in doc_ids_by_press.values():
    assert ids["candidate_doc_ids"] <= ids["rule_doc_ids"]
    assert ids["candidate_doc_ids"] <= ids["error_history_doc_ids"]
    assert not (ids["target_sentence_ids"] & ids["martial_law_sentence_ids"])

review_summary = pd.DataFrame([
    {
        "press": press,
        "source_articles": ids["source_doc_count"],
        "source_sentences": ids["source_sentence_count"],
        "rule_articles_detected": len(ids["rule_doc_ids"]),
        "martial_law_articles_excluded": len(ids["martial_law_doc_ids"]),
        "martial_law_article_ratio": (
            len(ids["martial_law_doc_ids"]) / ids["source_doc_count"]
            if ids["source_doc_count"]
            else 0
        ),
        "martial_law_article_pct": (
            len(ids["martial_law_doc_ids"]) * 100 / ids["source_doc_count"]
            if ids["source_doc_count"]
            else 0
        ),
        "target_sentences": ids["target_sentence_count"],
        "error_history_articles": len(ids["error_history_doc_ids"]),
        "candidate_articles": len(ids["candidate_doc_ids"]),
        "candidate_article_ratio": (
            len(ids["candidate_doc_ids"]) / ids["source_doc_count"]
            if ids["source_doc_count"]
            else 0
        ),
        "candidate_article_pct": (
            len(ids["candidate_doc_ids"]) * 100 / ids["source_doc_count"]
            if ids["source_doc_count"]
            else 0
        ),
        "target_sentence_ratio": (
            ids["target_sentence_count"] / ids["source_sentence_count"]
            if ids["source_sentence_count"]
            else 0
        ),
        "target_sentence_pct": (
            ids["target_sentence_count"] * 100 / ids["source_sentence_count"]
            if ids["source_sentence_count"]
            else 0
        ),
        "source_reviewed_candidate_articles": len(ids["reviewed_candidate_doc_ids"]),
        "reviewed_ratio": (
            len(ids["reviewed_candidate_doc_ids"])
            / len(ids["candidate_doc_ids"])
            if ids["candidate_doc_ids"]
            else 0
        ),
        "reviewed_pct": (
            len(ids["reviewed_candidate_doc_ids"]) * 100
            / len(ids["candidate_doc_ids"])
            if ids["candidate_doc_ids"]
            else 0
        ),
    }
    for press, ids in doc_ids_by_press.items()
])
overall = review_summary.sum(numeric_only=True)
overall["press"] = "전체"
overall["candidate_article_ratio"] = (
    overall["candidate_articles"] / overall["source_articles"]
    if overall["source_articles"]
    else 0
)
overall["candidate_article_pct"] = overall["candidate_article_ratio"] * 100
overall["martial_law_article_ratio"] = (
    overall["martial_law_articles_excluded"] / overall["source_articles"]
    if overall["source_articles"]
    else 0
)
overall["martial_law_article_pct"] = overall["martial_law_article_ratio"] * 100
overall["target_sentence_ratio"] = (
    overall["target_sentences"] / overall["source_sentences"]
    if overall["source_sentences"]
    else 0
)
overall["target_sentence_pct"] = overall["target_sentence_ratio"] * 100
overall["source_reviewed_candidate_articles"] = sum(
    len(ids["reviewed_candidate_doc_ids"])
    for ids in doc_ids_by_press.values()
)
overall["reviewed_ratio"] = (
    overall["source_reviewed_candidate_articles"] / overall["candidate_articles"]
    if overall["candidate_articles"]
    else 0
)
overall["reviewed_pct"] = overall["reviewed_ratio"] * 100
review_summary = pd.concat([review_summary, pd.DataFrame([overall])], ignore_index=True)

martial_law_article_ids = pd.DataFrame([
    {"press": press, "doc_id": doc_id}
    for press, ids in doc_ids_by_press.items()
    for doc_id in sorted(ids["martial_law_doc_ids"])
])
reviewed_id_trace = pd.DataFrame([
    {
        "press": press,
        "doc_id": doc_id,
        "reviewed": doc_id in ids["reviewed_candidate_doc_ids"],
    }
    for press, ids in doc_ids_by_press.items()
    for doc_id in sorted(ids["candidate_doc_ids"])
])

display(review_summary)
display(martial_law_article_ids)

,press,source_articles,source_sentences,rule_articles_detected,martial_law_articles_excluded,martial_law_article_ratio,martial_law_article_pct,target_sentences,error_history_articles,candidate_articles,candidate_article_ratio,candidate_article_pct,target_sentence_ratio,target_sentence_pct,source_reviewed_candidate_articles,reviewed_ratio,reviewed_pct
0,강원일보,14126.0,165006.0,13884.0,503.0,0.035608,3.56081,6145.0,3421.0,3413.0,0.241611,24.161121,0.037241,3.724107,303,0.088778,8.87782
1,전체,14126.0,165006.0,13884.0,503.0,0.035608,3.56081,6145.0,3421.0,3413.0,0.241611,24.161121,0.037241,3.724107,303,0.088778,8.87782


,press,doc_id
0,강원일보,NLRW2600000001.10010
1,강원일보,NLRW2600000001.10014
2,강원일보,NLRW2600000001.10016
3,강원일보,NLRW2600000001.10040
4,강원일보,NLRW2600000001.101
...,...,...
498,강원일보,NLRW2600000001.9755
499,강원일보,NLRW2600000001.977
500,강원일보,NLRW2600000001.9848
501,강원일보,NLRW2600000001.9873


- 룰 검출 기사를 제외하지 않고, 룰 검출 문장 중 휴먼 검수 후보 문장을 선별
- `EXT_E001`이면서 계엄(`EXT_S001`)이 아닌 문장을 포함한 기사만 유지
- 원천 `review_needed.jsonl`은 전처리/복원 검수 이력으로 별도 집계하며, 2차 LLM 오류 결과와 혼동하지 않음

In [16]:
filtered_articles = {
    press: select_target_documents(
        DATA_DIR / f"{press}.json",
        ids["candidate_doc_ids"],
    )
    for press, ids in doc_ids_by_press.items()
}

article_counts = {
    press: len(data["document"])
    for press, data in filtered_articles.items()
}
article_counts

{'강원일보': 3413}

In [17]:
# 필요하면 언론사별 필터 결과를 JSON으로 저장합니다.
OUTPUT_DIR = DATA_DIR / "filtered_by_rule_pipeline"
OUTPUT_DIR.mkdir(exist_ok=True)

for press, data in filtered_articles.items():
    output_path = OUTPUT_DIR / f"{press}.json"
    output_path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

review_summary.to_csv(OUTPUT_DIR / "review_summary.csv", index=False, encoding="utf-8-sig")
martial_law_article_ids.to_csv(OUTPUT_DIR / "martial_law_article_ids.csv", index=False, encoding="utf-8-sig")
reviewed_id_trace.to_csv(OUTPUT_DIR / "reviewed_id_trace.csv", index=False, encoding="utf-8-sig")

OUTPUT_DIR

WindowsPath('C:/work/ner/data/org/all_pass_ner_tagged_1/filtered_by_rule_pipeline')